In [0]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/finops_catalog_2/finops/raw/cloud_budget_2023_dataset.csv")
)

In [0]:
# df.display()
# df.printSchema()

In [0]:
df = spark.read.table(
    "finops_catalog.finops.gold_anomaly_features"
)

# display(df)

### Dataset Shape

In [0]:
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")

### Schema

In [0]:
df.printSchema()

### Column Names

In [0]:
for col in df.columns:
    print(col, end=", ")

### Summary Statistics

In [0]:
display(df.describe())

In [0]:
display(df.groupby("budget_status").count())

### Missing Values Analysis

In [0]:
from pyspark.sql.functions import *

null_counts = df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]
)

display(null_counts)

### Duplicate Records

In [0]:
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()

print("Total Rows:", total_rows)
print("Distinct Rows:", distinct_rows)
print("Duplicate Rows:", total_rows - distinct_rows)

### Cost Distribution

### Daily Cost Trend

In [0]:
from pyspark.sql.functions import sum

daily_cost = (
    df.groupBy("date")
      .agg(
          round(sum("net_cost"),2).alias("total_cost")
      )
      .orderBy("date")
)

display(daily_cost)

### Business Unit Cost Analysis

In [0]:
from pyspark.sql.functions import sum

business_cost = (
    df.groupBy("business_unit")
      .agg(
          round(sum("net_cost"),2).alias("total_cost")
      )
      .orderBy(col("total_cost").desc())
)

display(business_cost)

### Department Spend Analysis

In [0]:
department_cost = (
    df.groupBy("department")
      .agg(
          round(sum("net_cost"),2).alias("total_cost")
      )
      .orderBy(col("total_cost").desc())
)

display(department_cost)

### Service Cost Analysis

In [0]:
service_cost = (
    df.groupBy("service")
      .agg(
          round(sum("net_cost"), 2)
          .alias("total_cost")
      )
      .orderBy(col("total_cost").desc())
)

display(service_cost)

### Budget Utilization Distribution

In [0]:
# display(
#     df.select("budget_utilization_pct")
# )

### Anomaly Distribution

In [0]:

# df.groupBy("is_anomaly").count().show()
display(
    df.select(
        "anomaly_score",
        "net_cost",
        "budget_utilization_pct"
    )
    .orderBy("anomaly_score", ascending=False)
)

### Cloud Provider Distribution

In [0]:
display(
    df.groupBy("cloud_provider")
      .count()
)

### Correlation Analysis

In [0]:
pdf = df.select(
    "net_cost",
    "usage_quantity",
    "budget_utilization_pct",
    "cost_variance_7d_pct",
    "cost_variance_30d_pct"
).toPandas()

### Correlation Matrix

In [0]:
pdf.corr()

In [0]:
import matplotlib.pyplot as plt

corr = pdf.corr()

plt.figure(figsize=(8,6))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.show()

### Top 10 Highest Cost Records

In [0]:
display(
    df.orderBy(
        col("net_cost").desc()
    )
    .limit(10)
)

### Budget Status Analysis

In [0]:

df.groupBy("budget_status").count().show()
